# World Cup Agent Arena — Workflow walkthrough

Step-by-step demo of the eight-step pre-match flow.
Run cells top-to-bottom.

**Before running**, replace the two placeholder credentials in the **Setup** cell:
`ARENA_KEY` (mint at https://staging.stair-ai.com/api-keys) and `ANTHROPIC_KEY`
(get one at https://console.anthropic.com). The Supabase URL and publishable key
are shared across every builder.



## SETUP



In [ ]:
import os, json, time, uuid, requests

ARENA            = "https://staging.stair-ai.com"
SPORTMONKS_PROXY = f"{ARENA}/api/v1/data/proxy/sportmonks/v3/football"
POLYMARKET_CLOB  = f"{ARENA}/api/v1/data/proxy/polymarket-clob"
POLYMARKET_GAMMA = f"{ARENA}/api/v1/data/proxy/polymarket-gamma"
ARENA_KEY        = "FILL IN YOUR ARENA KEY HERE"
# Staging shares a single publishable Supabase key for every builder — no
# per-account JWT, no extra setup. The arena will publish these two values
# alongside the API key minted in the portal.
SUPABASE         = "https://ezvbmtvrvzageqixvdak.supabase.co"
SUPABASE_KEY     = "sb_publishable__m8bOkD05ToFwATpaWST5w_2-3fGS7V"
ANTHROPIC_KEY    = "FILL IN YOUR ANTHROPIC KEY HERE"
H_ARENA          = {"x-api-key": ARENA_KEY}
H_WCA            = {"apikey": SUPABASE_KEY, "Accept-Profile": "world_cup_arena"}

# Tournament constant — WC2026 is the only season this guide targets.
SPORTMONKS_SEASON_ID = 26618

# Reasoning-Ledger schema constants (per schema/records.schema.json v0.3 in
# StairAI/Reasoning-Ledger). agent_id is NOT set client-side: the arena
# resolves it server-side from the x-api-key on POST, so the wire records
# omit it. The local dump produced by this script also omits it for fidelity
# with what the agent actually transmits.
LEDGER_SCHEMA_VERSION = "0.3"
LLM_MODEL             = "claude-haiku-4-5-20251001"

# Anthropic extended-thinking knobs. budget_tokens must be < max_tokens; when
# enabled, response.content contains both `thinking` and `text` blocks — see
# scripts/model_reasoning_blocks.ipynb (Pattern A) for the canonical reference.
LLM_MAX_TOKENS      = 2400
LLM_THINKING_BUDGET = 1024
LLM_THINKING        = {"type": "enabled", "budget_tokens": LLM_THINKING_BUDGET}


def _extract(resp):
    """Pull the final text and (concatenated) thinking blocks from an Anthropic
    response with extended thinking enabled."""
    text_parts, thinking_parts = [], []
    for block in resp.content:
        if block.type == "thinking":
            thinking_parts.append(block.thinking)
        elif block.type == "text":
            text_parts.append(block.text)
    return "".join(text_parts), "\n\n".join(thinking_parts)


## STEP 1 · List fixtures (Sportmonks schedules)



GET the season schedule — every stage, round, and fixture for WC2026.
Use this to discover which fixtures exist and pick the one to reason about.
Original Sportmonks path:
https://api.sportmonks.com/v3/football/schedules/seasons/{SEASON_ID}?api_token=…
Orignal Sportmonks doc:
https://docs.sportmonks.com/v3/world-cup-2026/how-to-build-your-world-cup-application



In [ ]:
r = requests.get(
    f"{SPORTMONKS_PROXY}/schedules/seasons/{SPORTMONKS_SEASON_ID}",
    headers=H_ARENA, timeout=10,
)
print(f"HTTP {r.status_code}  ·  Content-Type: {r.headers.get('Content-Type')}")
print(f"Response-headers (arena/quota): "
      f"{ {k: v for k, v in r.headers.items() if k.lower().startswith('x-arena')} }")
print(f"=== response body (first 2 KB) ===\n{r.text[:2000]}\n=== end body ===")
r.raise_for_status()

envelope = r.json()
print(f"envelope keys: {list(envelope.keys())}")
# Staging wraps the Sportmonks response in {body, duration, statusCode, requestId, _proxy, headers}.
# Doc says proxy is verbatim — TODO: reconcile. For now, peel the envelope.
body = envelope["body"]
schedule = body["data"]
print(f"type(schedule) = {type(schedule).__name__}  ·  len = {len(schedule)}")
print()


# Inspecting the schedule above: the opener is Mexico (MEX, country_id 458)
# vs South Africa (ZAF, country_id 146), 2026-06-11 19:00, fixture_id 19609127.
SPORTMONKS_FIXTURE_ID = 19609127


### Polymarket event slug for the chosen fixture (from arena /web/mapping)

The arena exposes a curated fixture↔polymarket-event mapping. The full row
carries the operator's snapshot of condition_ids / token_yes / confidence,
but the agent only needs `polymarket_event_slug` — everything else gets
re-fetched live from Polymarket in step 3.



In [ ]:
r = requests.get(
    f"{ARENA}/api/v1/web/mapping",
    params={"fixture_id": SPORTMONKS_FIXTURE_ID},
    headers=H_ARENA, timeout=10,
)
r.raise_for_status()
mappings = r.json().get("mappings") or []
polymarket_event_slug = mappings[0]["polymarket_event_slug"] if mappings else None
print()
print(f"=== polymarket_event_slug: {polymarket_event_slug!r} ===")


## STEP 2 · Sportmonks fixture detail



Pull the chosen fixture's pre-match prediction inputs.
`statistics` only fills post-match, so swap it for actually-useful includes:
predictions — Sportmonks' own ML model probabilities
odds        — bookmaker prices (we'll average to a consensus probability)
xGFixture   — expected-goals projection
Original Sportmonks doc:
https://docs.sportmonks.com/v3/endpoints-and-entities/endpoints/fixtures/get-fixture-by-id



In [ ]:
r = requests.get(
    f"{SPORTMONKS_PROXY}/fixtures/{SPORTMONKS_FIXTURE_ID}",
    params={"include": "participants;predictions;odds;xGFixture"},
    headers=H_ARENA, timeout=60,
)
print(f"HTTP {r.status_code}")
print(f"=== response body (first 3 KB) ===\n{r.text[:3000]}\n=== end body ===")
r.raise_for_status()
fixture = r.json()["body"]["data"]    # same envelope as step 1
print(f"fixture top-level keys: {list(fixture.keys())}")

home = next(p for p in fixture["participants"] if p["meta"]["location"] == "home")
away = next(p for p in fixture["participants"] if p["meta"]["location"] == "away")


### LLM digest of Sportmonks' pre-match inputs

Distill the rich (and noisy) Sportmonks payload into a compact JSON the later
combined-analysis step (LLM #1 in §5) can consume alongside Supabase H2H and
Polymarket mids.



In [ ]:
import anthropic
client = anthropic.Anthropic(api_key=ANTHROPIC_KEY)      # reads ANTHROPIC_API_KEY from env

DIGEST_SYS = (
    "You are a soccer analyst. You receive a raw Sportmonks pre-match payload for "
    "one fixture and must distil it into a self-contained JSON digest that a "
    "downstream LLM (with no other context about Sportmonks) will read.\n\n"

    "## Input shape\n"
    "  - fixture       : match name (e.g. 'Mexico vs South Africa')\n"
    "  - home_code     : home team short code (use as a JSON key for the home outcome)\n"
    "  - away_code     : away team short code (use as a JSON key for the away outcome)\n"
    "  - predictions[] : Sportmonks ML model rows. Each row has `type_id` (numeric — "
    "                    the Full-Time-Result / 1X2 winner type carries win/draw/loss "
    "                    probabilities) and a `predictions` object with the numeric "
    "                    probability values. May be empty if Sportmonks has no model "
    "                    output for this fixture.\n"
    "  - odds[]        : bookmaker odds rows. Each row is ONE bookmaker's price for "
    "                    ONE outcome of ONE market. Key fields: `bookmaker_id`, "
    "                    `market_id` (1 = Full-Time-Result / 1X2 winner — IGNORE other "
    "                    markets), `label` ('1' home, 'X' draw, '2' away), `value` "
    "                    (decimal odds), `probability` (implied probability as a "
    "                    percentage 0-100). For the consensus, average `probability` "
    "                    across all bookmakers for market_id == 1 only, then divide by "
    "                    100 to express as 0..1. May be empty.\n"
    "  - xGFixture[]   : expected-goals entries per team. Each row has "
    "                    `participant_id` (team id matching home/away participant) and "
    "                    `value` (xG number). May be empty.\n\n"

    "## Output schema (return ONLY this JSON — no prose, no code fences)\n"
    "{\n"
    "  'fixture'                       : str,                                                          // echo input\n"
    "  'home_team'                     : str,                                                          // home_code\n"
    "  'away_team'                     : str,                                                          // away_code\n"
    "  'sportmonks_ml_win_prob'        : {home_code: float, 'draw': float, away_code: float} | null,   // probabilities in 0..1; sum ≈ 1\n"
    "  'bookmaker_consensus_win_prob'  : {home_code: float, 'draw': float, away_code: float} | null,   // probabilities in 0..1\n"
    "  'bookmaker_count'               : int | null,                                                   // bookmakers averaged into consensus\n"
    "  'expected_goals'                : {home_code: float, away_code: float} | null,                  // xG per side\n"
    "  'data_availability': {                                                                          // honest reporting so downstream knows what's missing\n"
    "    'sportmonks_ml'        : 'available' | 'missing',\n"
    "    'bookmaker_consensus'  : 'available' | 'missing',\n"
    "    'expected_goals'       : 'available' | 'missing'\n"
    "  },\n"
    "  'summary': str   // 1-3 sentences. MUST be readable in isolation by an LLM that has no other Sportmonks context. Name the available signals and what they imply; if everything is missing, say so plainly. Mention the home/away team codes by name.\n"
    "}\n\n"

    "Use null (not 0) when source data is missing. Do NOT fabricate values."
)

llm_digest = client.messages.create(
    model=LLM_MODEL,
    max_tokens=LLM_MAX_TOKENS,
    thinking=LLM_THINKING,
    system=DIGEST_SYS,
    messages=[{
        "role": "user",
        "content": json.dumps({
            "fixture":     fixture["name"],
            "home_code":   home["short_code"],
            "away_code":   away["short_code"],
            "predictions": fixture.get("predictions"),
            "odds":        fixture.get("odds"),
            "xGFixture":   fixture.get("xgfixture"),    # field is lowercase despite include name
        }),
    }],
)
raw, thinking_digest = _extract(llm_digest)
print()
print("=== LLM raw output ===")
print(raw)
print(f"--- thinking ({len(thinking_digest)} chars) ---")
print(thinking_digest[:400] + ("…" if len(thinking_digest) > 400 else ""))

# Strip ```json fences / prose if the model wrapped its answer.
import re
match = re.search(r"\{.*\}", raw, re.DOTALL)
sportmonks_digest = json.loads(match.group(0)) if match else None
print()
print("=== Sportmonks digest (parsed) ===")
print(json.dumps(sportmonks_digest, indent=2))


## STEP 3 · Polymarket market + mids



Resolve the 3-way moneyline market from Polymarket itself:
3a · Gamma /events?slug=<slug>  → event with three nested markets,
each carrying conditionId + clobTokenIds (YES + NO token ids)
3b · For each YES token, CLOB /midpoint → live mid price
Then assemble a single `moneyline` dict for the digest LLM.

Per Polymarket's WC2026 convention, an event's ticker is
fifwc-{home_code}-{away_code}-{YYYY-MM-DD}
and each child market's slug is the ticker + "-{outcome_code|draw}".



In [ ]:
import re
TICKER_RE = re.compile(r"^fifwc-([a-z]{2,4})-([a-z]{2,4})-(\d{4}-\d{2}-\d{2})$")


def _clob_mid(token_id_str: str | None) -> float | None:
    """Single CLOB midpoint call. Polymarket's CLOB takes the token id as a
    decimal string (the raw value is a 78-digit integer)."""
    if not token_id_str:
        return None
    try:
        resp = requests.get(
            f"{POLYMARKET_CLOB}/midpoint",
            params={"token_id": token_id_str},
            headers=H_ARENA, timeout=10,
        )
        if not resp.ok:
            return None
        body = resp.json().get("body")
        if isinstance(body, dict) and "mid" in body:
            return float(body["mid"])
    except Exception:
        pass
    return None


def _outcome_from_market_slug(market_slug: str, ticker: str,
                              home_code: str, away_code: str) -> str | None:
    """Map a child-market slug ('fifwc-mex-rsa-2026-06-11-mex') to an
    outcome key ('home' | 'draw' | 'away')."""
    if not market_slug.startswith(ticker + "-"):
        return None
    suffix = market_slug[len(ticker) + 1:]
    if suffix == home_code: return "home"
    if suffix == "draw":    return "draw"
    if suffix == away_code: return "away"
    return None


if not polymarket_event_slug:
    moneyline = None
else:
    # 3a · Gamma: one call returns the event + its 3 child markets.
    r = requests.get(
        f"{POLYMARKET_GAMMA}/events",
        params={"slug": polymarket_event_slug},
        headers=H_ARENA, timeout=15,
    )
    r.raise_for_status()
    events = r.json().get("body") or []
    event  = events[0] if events else None

    if event is None:
        moneyline = None
    else:
        ticker = (event.get("ticker") or "").lower()
        m = TICKER_RE.match(ticker)
        if not m:
            moneyline = None
        else:
            pm_home_code, pm_away_code, _ = m.groups()
            outcomes: dict[str, dict] = {}
            for mkt in (event.get("markets") or []):
                key = _outcome_from_market_slug((mkt.get("slug") or "").lower(),
                                                ticker, pm_home_code, pm_away_code)
                if key is None:
                    continue
                # clobTokenIds is a JSON-encoded string: [YES_token, NO_token].
                try:
                    token_ids = json.loads(mkt.get("clobTokenIds") or "[]")
                except json.JSONDecodeError:
                    token_ids = []
                token_yes = token_ids[0] if token_ids else None
                outcomes[key] = {
                    "team_code":       key if key == "draw" else (
                                            pm_home_code.upper() if key == "home"
                                            else pm_away_code.upper()),
                    "condition_id":    mkt.get("conditionId"),
                    "token_yes":       token_yes,
                    "current_mid_yes": _clob_mid(token_yes),  # 3b · one CLOB call per YES token
                }

            moneyline = {
                "sportmonks_match_id":   SPORTMONKS_FIXTURE_ID,
                "fixture":               event.get("title"),
                "kickoff_utc":           event.get("startDate"),
                "polymarket_event_slug": polymarket_event_slug,
                "outcomes":              outcomes,
            }

print()
print("=== moneyline market response ===")
print(json.dumps(moneyline, indent=2, default=str))


### LLM digest of the moneyline market

Mirrors STEP 2's pattern: distil the raw response into a self-contained JSON
that the downstream combined-analysis LLM can consume without any other
Polymarket context.



In [ ]:
POLYMARKET_DIGEST_SYS = (
    "You are an analyst digesting a Polymarket moneyline (3-way match-winner) "
    "market response into a self-contained JSON for a downstream LLM that has "
    "no other Polymarket context.\n\n"

    "## Input shape\n"
    "  - sportmonks_match_id   : numeric fixture id (echo)\n"
    "  - fixture               : match name (e.g. 'Mexico vs South Africa')\n"
    "  - kickoff_utc           : ISO kickoff timestamp\n"
    "  - polymarket_event_slug : Polymarket event slug grouping the 3 binary markets\n"
    "  - outcomes.{home,draw,away}\n"
    "      team_code           : team short code (or 'draw' for the draw outcome)\n"
    "      condition_id        : Polymarket condition id (needed for trade execution)\n"
    "      token_yes           : ERC1155 YES-side token id (buy YES to back the outcome)\n"
    "      current_mid_yes     : midpoint price of the YES token in 0..1 == implied probability\n"
    "                            of that outcome winning. null if CLOB lookup failed.\n\n"

    "## Output schema (return ONLY this JSON — no prose, no code fences)\n"
    "{\n"
    "  'fixture'              : str,\n"
    "  'market_handle'        : str,                                                          // polymarket_event_slug\n"
    "  'implied_win_prob'     : {home_code: float, 'draw': float, away_code: float} | null,   // from current_mid_yes; null if unavailable\n"
    "  'sum_implied_prob'     : float | null,                                                 // should be ≈1.0; outside [0.95, 1.10] = stale prices or arb gap\n"
    "  'execution_handles'    : {home_code: {condition_id, token_yes},                        // for the downstream trade-execution step\n"
    "                            'draw'   : {condition_id, token_yes},\n"
    "                            away_code: {condition_id, token_yes}},\n"
    "  'data_availability'    : 'mids_available' | 'mids_partial' | 'mids_missing' | 'no_market',\n"
    "  'summary'              : str   // 1-3 sentences self-contained. Name the favorite (highest implied prob), the spread, and any anomaly. If mids are missing, say so plainly and identify what's still available (execution handles can still be used to place orders blind).\n"
    "}\n\n"

    "Use null when input shows null. Do NOT fabricate prices."
)

if moneyline is None:
    polymarket_digest = {
        "fixture":              None,
        "market_handle":        None,
        "implied_win_prob":     None,
        "sum_implied_prob":     None,
        "execution_handles":    None,
        "data_availability":    "no_market",
        "summary":              f"No Polymarket moneyline mapping for Sportmonks fixture "
                                f"{SPORTMONKS_FIXTURE_ID}. The fixture either isn't listed on "
                                f"Polymarket yet or its curated mapping is marked no_match.",
    }
else:
    llm_pm = client.messages.create(
        model=LLM_MODEL,
        max_tokens=LLM_MAX_TOKENS,
        thinking=LLM_THINKING,
        system=POLYMARKET_DIGEST_SYS,
        messages=[{"role": "user", "content": json.dumps(moneyline)}],
    )
    raw_pm, thinking_pm = _extract(llm_pm)
    print()
    print("=== Polymarket digest raw output ===")
    print(raw_pm)
    print(f"--- thinking ({len(thinking_pm)} chars) ---")
    print(thinking_pm[:400] + ("…" if len(thinking_pm) > 400 else ""))
    m = re.search(r"\{.*\}", raw_pm, re.DOTALL)
    polymarket_digest = json.loads(m.group(0)) if m else None

print()
print("=== Polymarket digest (parsed) ===")
print(json.dumps(polymarket_digest, indent=2))


## STEP 4 · Supabase aggregated data



Three sub-steps:
(4a) discover — read public.catalog_full to learn what tables exist
(4b) fetch    — pull relevant priors for both teams from chosen tables
(4c) digest   — LLM aggregates into a self-contained JSON for §5

Cross-system country_id quirk: Sportmonks numbers Mexico=458 / ZAF=146,
but world_cup_arena.dim_country (StatsBomb-derived) uses 147 / 211.



In [ ]:
TEAM_A_ID = 147   # Mexico
TEAM_B_ID = 211   # South Africa

H_PUBLIC = {"apikey": SUPABASE_KEY}                                       # default schema = public
H_WCA    = {"apikey": SUPABASE_KEY, "Accept-Profile": "world_cup_arena"}  # data layer


### 4a · DISCOVER

An agent that's new to the data layer queries the catalog first. This is the
only call needed to learn what's available — names, categories, row counts,
table descriptions, and (via catalog_columns) every column's data type +
meaning. No external docs required.



In [ ]:
print("--- 4a · discover available tables ---")
r = requests.get(
    f"{SUPABASE}/rest/v1/catalog_full",
    params={
        "select": "table_name,category,row_count,table_description",
        "order":  "category,table_name",
    },
    headers=H_PUBLIC, timeout=10,
)
r.raise_for_status()
catalog = r.json()
print(f"  found {len(catalog)} public tables:")
for t in catalog:
    desc = (t.get("table_description") or "—").replace("\n", " ")[:60]
    cat  = t.get("category") or "?"
    print(f"    [{cat:11s}] {t['table_name']:30s}  rows={t['row_count']:5d}  desc: {desc}")

# For the example we only fetch ONE priors table — `ads_a_country_style`,
# picked from the catalog above for its playing-style indicators. A real
# agent could pull more (H2H, KO pattern, special-match, etc.) — same
# pattern, just more rows in the dict.
WANTED_TABLE = "ads_a_country_style"


### 4b · FETCH



In [ ]:
print()
print(f"--- 4b · fetch {WANTED_TABLE} for both teams ---")
r = requests.get(
    f"{SUPABASE}/rest/v1/{WANTED_TABLE}",
    params={"country_id": f"in.({TEAM_A_ID},{TEAM_B_ID})", "select": "*"},
    headers=H_WCA, timeout=10,
)
r.raise_for_status()
priors_rows = r.json()
print(f"  {len(priors_rows)} row(s) returned")
print(f"  raw: {json.dumps(priors_rows, indent=2)[:600]}")


### 4c · DIGEST



In [ ]:
print()
print("--- 4c · LLM digest of supabase priors ---")
SUPABASE_DIGEST_SYS = (
    "You are an analyst aggregating Supabase priors data for one fixture into "
    "a self-contained JSON digest for a downstream LLM that has no other "
    "context about the data layer.\n\n"

    "## Input shape\n"
    "  - fixture        : match name\n"
    "  - source_table   : the Supabase table the rows came from (echo for traceability)\n"
    "  - home_code, away_code : team short codes (use as JSON keys for the output)\n"
    "  - home_id,   away_id   : country ids in this dataset (StatsBomb numbering)\n"
    "  - rows           : list of rows from `ads_a_country_style`, one per country.\n"
    "      Key columns include: country_id, set_piece_shots, set_piece_goals,\n"
    "      conversion_rate, group_matches, group_goals_against, ko_matches,\n"
    "      ko_goals_against, group_gpg (goals/game in group stage),\n"
    "      ko_gpg (goals/game in knockout stage).\n"
    "      Match each row to home_id / away_id by country_id. Sample sizes are\n"
    "      often tiny in this dataset — call that out if it impacts confidence.\n\n"

    "## Output schema (return ONLY this JSON — no prose, no code fences)\n"
    "{\n"
    "  'fixture'      : str,\n"
    "  'source_table' : str,                                              // echo\n"
    "  'teams': {\n"
    "    home_code: {                                                     // team A profile from country_style\n"
    "      'set_piece_efficiency' : float | null,                          // set_piece_goals / set_piece_shots\n"
    "      'set_piece_sample'     : int   | null,                          // set_piece_shots — proxies confidence\n"
    "      'group_goals_per_game' : float | null,\n"
    "      'ko_goals_per_game'    : float | null\n"
    "    },\n"
    "    away_code: { same shape }\n"
    "  },\n"
    "  'data_availability': 'rich' | 'partial' | 'sparse',\n"
    "  'summary': str   // 1-3 sentences self-contained. Name the two teams, the strongest\n"
    "                 // style signal, and call out small-sample caveats. Do NOT give a\n"
    "                 // win probability — that's for §5 combined analysis.\n"
    "}\n\n"

    "Use null when input is empty/missing. Don't fabricate values."
)

llm_sb = client.messages.create(
    model=LLM_MODEL,
    max_tokens=LLM_MAX_TOKENS,
    thinking=LLM_THINKING,
    system=SUPABASE_DIGEST_SYS,
    messages=[{
        "role": "user",
        "content": json.dumps({
            "fixture":      fixture["name"],
            "source_table": WANTED_TABLE,
            "home_code":    home["short_code"],
            "away_code":    away["short_code"],
            "home_id":      TEAM_A_ID,
            "away_id":      TEAM_B_ID,
            "rows":         priors_rows,
        }, default=str),
    }],
)
raw_sb, thinking_sb = _extract(llm_sb)
print("=== Supabase digest raw output ===")
print(raw_sb)
print(f"--- thinking ({len(thinking_sb)} chars) ---")
print(thinking_sb[:400] + ("…" if len(thinking_sb) > 400 else ""))

m = re.search(r"\{.*\}", raw_sb, re.DOTALL)
supabase_digest = json.loads(m.group(0)) if m else None
print()
print("=== Supabase digest (parsed) ===")
print(json.dumps(supabase_digest, indent=2))


## STEP 5 · LLM #1 — predict

Combine the Sportmonks and Supabase digests into the agent's own outcome
prediction. Polymarket is DELIBERATELY excluded here so this view is
formed independently of the market — comparing the two in §6 is where the
edge (if any) shows up.



In [ ]:
PREDICT_SYS = (
    "You are a soccer match analyst. You receive two pre-distilled digests for "
    "one fixture and must produce the agent's own outcome prediction.\n\n"

    "## Input shape\n"
    "  - fixture           : match name\n"
    "  - home_code         : home team short code (use as a JSON key for the home outcome)\n"
    "  - away_code         : away team short code (use as a JSON key for the away outcome)\n"
    "  - sportmonks_digest : digest of Sportmonks pre-match data (model probs, bookmaker\n"
    "                        consensus, expected goals). Values may be null when staging\n"
    "                        hasn't seeded the data — see its `data_availability` flag.\n"
    "  - supabase_digest   : digest of long-horizon priors (playing style, set-piece\n"
    "                        efficiency, group/KO goals-per-game). Note its sample-size\n"
    "                        caveats.\n\n"

    "## Output schema (return ONLY this JSON — no prose, no code fences)\n"
    "{\n"
    "  'fixture'    : str,\n"
    "  'outcome'    : str,                          // home_code | 'draw' | away_code — the most likely outcome\n"
    "  'probability': float,                        // 0..1; confidence in `outcome`\n"
    "  'rationale'  : str,                          // 1-3 sentences self-contained. Name the teams,\n"
    "                                               // the signals you leaned on, and major caveats.\n"
    "  'used_signals': {                            // for traceability into §6\n"
    "    'sportmonks' : 'leaned_on' | 'unavailable',\n"
    "    'supabase'   : 'leaned_on' | 'unavailable'\n"
    "  },\n"
    "  'confidence_level': 'high' | 'medium' | 'low'   // honest about how thin your evidence was\n"
    "}\n\n"

    "Be honest about uncertainty: if both digests are sparse, low confidence is the right answer. "
    "Do NOT consult the market (you don't have it). Probability must reflect what the priors say "
    "alone — anchoring to a market mid would defeat the point."
)

llm_predict = client.messages.create(
    model=LLM_MODEL,
    max_tokens=LLM_MAX_TOKENS,
    thinking=LLM_THINKING,
    system=PREDICT_SYS,
    messages=[{
        "role": "user",
        "content": json.dumps({
            "fixture":           fixture["name"],
            "home_code":         home["short_code"],
            "away_code":         away["short_code"],
            "sportmonks_digest": sportmonks_digest,
            "supabase_digest":   supabase_digest,
        }),
    }],
)
raw_pred, thinking_pred = _extract(llm_predict)
print()
print("=== STEP 5 · prediction raw output ===")
print(raw_pred)
print(f"--- thinking ({len(thinking_pred)} chars) ---")
print(thinking_pred[:400] + ("…" if len(thinking_pred) > 400 else ""))
m = re.search(r"\{.*\}", raw_pred, re.DOTALL)
prediction = json.loads(m.group(0)) if m else None
print()
print("=== prediction (parsed) ===")
print(json.dumps(prediction, indent=2))


## STEP 6 · LLM #2 — strategy

Combine the agent's prediction (§5) with the Polymarket digest (§3) into
a concrete trade decision. This is where edge (agent_prob − market_prob)
meets confidence and bankroll discipline.



In [ ]:
STRATEGY_SYS = (
    "You are a bankroll manager for a $100 demo account. You receive the agent's "
    "own prediction and the current Polymarket market view, and decide whether "
    "to trade and on what terms.\n\n"

    "## Input shape\n"
    "  - prediction        : {outcome, probability, confidence_level, rationale, …}\n"
    "                        The agent's own view, formed without seeing the market.\n"
    "  - polymarket_digest : {implied_win_prob, sum_implied_prob, execution_handles,\n"
    "                        market_handle, data_availability, summary}.\n"
    "                        The market's view (implied_win_prob keys match team codes).\n\n"

    "## How to decide\n"
    "  1. Edge = prediction.probability − polymarket_digest.implied_win_prob[prediction.outcome]\n"
    "     (in percentage points). Positive edge = market UNDER-prices the pick → long.\n"
    "     Negative edge = market OVER-prices the pick → short.\n"
    "  2. Size discipline (max $5 per trade, $100 wallet):\n"
    "       |edge| < 5pp                  → don't trade (noise)\n"
    "       |edge| 5-15pp                 → $1-2  (modest position)\n"
    "       |edge| > 15pp                 → $3-5  (high-conviction position)\n"
    "     Then HALVE the size if confidence_level is 'low'.\n"
    "       confidence 'medium'           → use the size above\n"
    "       confidence 'high'             → use up to 1.5× (capped at $5)\n"
    "     If the Polymarket digest's data_availability is not 'mids_available', skip — you\n"
    "     can't price an edge without mids.\n"
    "  3. limit_price is the worst price per share you'll accept. For long, a bit ABOVE the\n"
    "     current mid for the YES token (e.g. mid 0.665 → limit 0.68). For short, a bit\n"
    "     ABOVE the current mid for the NO token, which is (1 − mid_yes) (e.g. mid_yes 0.665\n"
    "     → NO_mid 0.335 → limit 0.36).\n\n"

    "## Output schema (return ONLY this JSON — no prose, no code fences)\n"
    "{\n"
    "  'should_trade'   : bool,\n"
    "  'outcome'        : str,                    // echo prediction.outcome — what to trade\n"
    "  'direction'      : 'long' | 'short',       // long = back the outcome; short = fade it\n"
    "  'size_usdc'      : float,                  // 0 when not trading; ≤5 for this demo\n"
    "  'limit_price'    : float,                  // 0..1; see rule 3 above\n"
    "  'edge_pp'        : float,                  // (agent_prob − market_prob) × 100\n"
    "  'market_handle'  : str,                    // echo polymarket_digest.market_handle for traceability\n"
    "  'rationale'      : str                     // 1-3 sentences self-contained: state the edge,\n"
    "                                             // the size logic, the limit_price logic.\n"
    "}\n\n"

    "Be conservative: small wallet, weak conviction → skipping is a valid answer."
)

llm_strategy = client.messages.create(
    model=LLM_MODEL,
    max_tokens=LLM_MAX_TOKENS,
    thinking=LLM_THINKING,
    system=STRATEGY_SYS,
    messages=[{
        "role": "user",
        "content": json.dumps({
            "prediction":        prediction,
            "polymarket_digest": polymarket_digest,
        }),
    }],
)
raw_strat, thinking_strat = _extract(llm_strategy)
print()
print("=== STEP 6 · strategy raw output ===")
print(raw_strat)
print(f"--- thinking ({len(thinking_strat)} chars) ---")
print(thinking_strat[:400] + ("…" if len(thinking_strat) > 400 else ""))
m = re.search(r"\{.*\}", raw_strat, re.DOTALL)
strategy = json.loads(m.group(0)) if m else None
print()
print("=== strategy (parsed) ===")
print(json.dumps(strategy, indent=2))


## STEP 7 · Open position

When the strategy LLM says don't trade, skip silently — predict-only runs
are a fully-supported flow (the trace still flows to the ledger in §8).
When the arena /orders endpoint isn't live yet on staging, the POST will
404. We print the request payload either way so the shape is auditable.



In [ ]:
if strategy and strategy.get("should_trade"):
    order_payload = {
        "fixture_code":           str(SPORTMONKS_FIXTURE_ID),     # canonical fixture handle (TBD whether arena keys by sportmonks_id or its own)
        "outcome":                strategy["outcome"],
        "direction":              strategy["direction"],
        "size_usdc":              f"{strategy['size_usdc']:.2f}",
        "limit_price":            strategy["limit_price"],
        "time_in_force_seconds":  30,
        "idempotency_key":        str(uuid.uuid4()),
    }
    print()
    print("=== STEP 7 · order payload ===")
    print(json.dumps(order_payload, indent=2))
    try:
        r = requests.post(
            f"{ARENA}/api/v1/orders",       # path TBD — adjust when arena ships /orders
            headers=H_ARENA, timeout=10,
            json=order_payload,
        )
        print(f"HTTP {r.status_code}  body: {r.text[:300]}")
        order_response = r.json() if r.ok else None
    except Exception as e:
        print(f"order POST failed: {type(e).__name__}: {e}")
        order_response = None
else:
    print()
    print(f"=== STEP 7 · skipped (strategy.should_trade={strategy and strategy.get('should_trade')}) ===")
    order_response = None


## STEP 8 · Report ledger trace

Schema-compliant records per StairAI/Reasoning-Ledger v0.3
(schema/records.schema.json on branch v3-update-enhance-thinking).
Built without the SDK — same payload shape, with x-api-key auth.

14-record DAG (15 when strategy.should_trade=True and an order Acting is appended):
Observing (trigger)
ToolCalling × 7  — sportmonks schedule / arena polymarket-slug mapping /
polymarket Gamma event / polymarket CLOB mids /
sportmonks fixture detail / supabase catalog / supabase priors
Thinking    × 5  — sportmonks digest / polymarket digest / supabase digest /
predict / strategy
Acting      × 1  — prediction (validated + scored by the arena;
behavior=Acting, action_type=prediction)
Acting      × 0..1 — order (open_order intent record; only when strategy.should_trade)



In [ ]:
LEDGER_SESSION_ID = f"prematch:{SPORTMONKS_FIXTURE_ID}"

def _new_record(behavior, **fields):
    """Compose the BaseRecord envelope + behavior-specific fields.

    Note: agent_id is intentionally omitted. The arena resolves it server-side
    from the x-api-key on POST, so wire records do not carry it. The local
    dump produced by this script mirrors that — schema-wise, agent_id is
    required, but it only becomes present after the server enriches the
    record."""
    rec = {
        "schema_version": LEDGER_SCHEMA_VERSION,
        "session_id":     LEDGER_SESSION_ID,
        "record_id":      str(uuid.uuid4()),
        "behavior":       behavior,
        "client_ts_utc":  int(time.time() * 1000),
    }
    rec.update({k: v for k, v in fields.items() if v is not None})
    return rec

def _mi(resp):
    """Build a ModelInvocation dict from an Anthropic response. Captures the
    `internal_reasoning` field (the model's extended-thinking trace) when
    present — see schema v0.3 §ModelInvocation."""
    _, thinking = _extract(resp)
    mi = {
        "provider":   "anthropic",
        "model_name": LLM_MODEL,
        "tokens_in":  resp.usage.input_tokens,
        "tokens_out": resp.usage.output_tokens,
    }
    if thinking:
        mi["internal_reasoning"] = thinking
    return mi

def _trunc(obj, limit=30000):
    """JSON-stringify + truncate to keep individual fields under SDK size limits
    (Thinking.output_payload ≤ 32 KB; per-record JSON ≤ 64 KB)."""
    s = obj if isinstance(obj, str) else json.dumps(obj, default=str)
    return s if len(s) <= limit else s[:limit] + f"…[truncated, was {len(s)} chars]"


# (1) Observing — synthetic cron trigger that woke the agent.
rec_trigger = _new_record(
    "Observing",
    trigger_source="dev-guide-workflow-test",
    trigger_type="cron_trigger",
    trigger_description=f"Pre-match prediction run for fixture {SPORTMONKS_FIXTURE_ID} ({fixture['name']})",
    trigger_payload_summary=(
        f"fixture_id={SPORTMONKS_FIXTURE_ID}; window=PRE_MATCH; "
        f"kickoff_utc={fixture['starting_at']}; home={home['short_code']}; away={away['short_code']}"
    ),
)

# (2) ToolCalling — Sportmonks schedule
rec_sm_schedule = _new_record(
    "ToolCalling",
    upstream_record_id=[rec_trigger["record_id"]],
    tool_meta={"name": "sportmonks", "endpoint": "/v3/football/schedules/seasons/{season_id}",
               "via": "arena.sportmonks_proxy"},
    description="List WC2026 season schedule to discover fixtures",
    input_payload={"season_id": 26618},
    output_payload={"stage_count": len(schedule), "picked_fixture_id": SPORTMONKS_FIXTURE_ID},
    success=True,
)

# (3) ToolCalling — Sportmonks fixture detail
rec_sm_fixture = _new_record(
    "ToolCalling",
    upstream_record_id=[rec_sm_schedule["record_id"]],
    tool_meta={"name": "sportmonks", "endpoint": "/v3/football/fixtures/{fixture_id}",
               "via": "arena.sportmonks_proxy"},
    description="Fetch fixture detail with pre-match prediction includes",
    input_payload={"fixture_id": SPORTMONKS_FIXTURE_ID,
                   "include":    "participants;predictions;odds;xGFixture"},
    output_payload={
        "fixture_name":      fixture["name"],
        "kickoff_utc":       fixture["starting_at"],
        "participants":      [{"id": p["id"], "name": p["name"],
                               "short_code": p["short_code"],
                               "country_id": p["country_id"],
                               "location": p["meta"]["location"]} for p in fixture["participants"]],
        "predictions_count": len(fixture.get("predictions") or []),
        "odds_count":        len(fixture.get("odds") or []),
        "xgfixture_count":   len(fixture.get("xgfixture") or []),
    },
    success=True,
)

# (4) Thinking — Sportmonks digest
rec_th_sportmonks = _new_record(
    "Thinking",
    upstream_record_id=[rec_sm_fixture["record_id"]],
    model_invocation=_mi(llm_digest),
    prompt=_trunc(DIGEST_SYS, limit=16000),
    inputs=[{
        "input_record_id": rec_sm_fixture["record_id"],
        "input_payload":   _trunc({
            "fixture":     fixture["name"],
            "home_code":   home["short_code"],
            "away_code":   away["short_code"],
            "predictions": fixture.get("predictions"),
            "odds":        fixture.get("odds"),
            "xGFixture":   fixture.get("xgfixture"),
        }),
    }],
    output_payload=_trunc(sportmonks_digest),
)

# (5a) ToolCalling — arena: look up the polymarket event slug for the fixture.
rec_pm_slug = _new_record(
    "ToolCalling",
    upstream_record_id=[rec_sm_schedule["record_id"]],
    tool_meta={"name": "arena-mapping",
               "endpoint": "/api/v1/web/mapping"},
    description="Look up curated Polymarket event_slug for this Sportmonks fixture",
    input_payload={"fixture_id": SPORTMONKS_FIXTURE_ID},
    output_payload={"polymarket_event_slug": polymarket_event_slug},
    success=polymarket_event_slug is not None,
)

# (5b) ToolCalling — Polymarket Gamma: fetch the event + nested markets
# (condition_ids + clobTokenIds for home / draw / away).
rec_pm_event = _new_record(
    "ToolCalling",
    upstream_record_id=[rec_pm_slug["record_id"]],
    tool_meta={"name": "polymarket-gamma",
               "endpoint": "/api/v1/data/proxy/polymarket-gamma/events",
               "via": "arena.proxy"},
    description="Fetch Polymarket event + 3 child winner markets by slug",
    input_payload={"slug": polymarket_event_slug},
    output_payload={
        "outcomes": {k: {"team_code":     moneyline["outcomes"][k]["team_code"],
                         "condition_id":  moneyline["outcomes"][k]["condition_id"],
                         "token_yes":     moneyline["outcomes"][k]["token_yes"]}
                     for k in moneyline["outcomes"]}
    } if moneyline else None,
    success=moneyline is not None,
)

# (5c) ToolCalling — Polymarket CLOB: live midpoint per YES token (3 calls
# summarized into one record).
rec_pm_mids = _new_record(
    "ToolCalling",
    upstream_record_id=[rec_pm_event["record_id"]],
    tool_meta={"name": "polymarket-clob",
               "endpoint": "/api/v1/data/proxy/polymarket-clob/midpoint",
               "via": "arena.proxy"},
    description="Fetch CLOB midpoint per outcome YES token (home / draw / away)",
    input_payload={"token_ids": [
        moneyline["outcomes"][k]["token_yes"] for k in moneyline["outcomes"]
    ] if moneyline else None},
    output_payload={
        k: moneyline["outcomes"][k]["current_mid_yes"] for k in moneyline["outcomes"]
    } if moneyline else None,
    success=moneyline is not None,
)

# (6) Thinking — Polymarket digest
rec_th_polymarket = _new_record(
    "Thinking",
    upstream_record_id=[rec_pm_slug["record_id"],
                        rec_pm_event["record_id"],
                        rec_pm_mids["record_id"]],
    model_invocation=_mi(llm_pm),
    prompt=_trunc(POLYMARKET_DIGEST_SYS, limit=16000),
    inputs=[{
        "input_record_id": rec_pm_mids["record_id"],
        "input_payload":   _trunc(moneyline),
    }],
    output_payload=_trunc(polymarket_digest),
)

# (7) ToolCalling — Supabase catalog discovery
rec_sb_catalog = _new_record(
    "ToolCalling",
    upstream_record_id=[rec_trigger["record_id"]],
    tool_meta={"name": "supabase", "endpoint": "/rest/v1/catalog_full"},
    description="Discover available Supabase tables via the public catalog",
    input_payload={"params": {"select": "table_name,category,row_count,table_description",
                              "order":  "category,table_name"}},
    output_payload={"available_tables": [t["table_name"] for t in catalog],
                    "count": len(catalog)},
    success=True,
)

# (8) ToolCalling — Supabase priors fetch
rec_sb_priors = _new_record(
    "ToolCalling",
    upstream_record_id=[rec_sb_catalog["record_id"], rec_sm_fixture["record_id"]],
    tool_meta={"name": "supabase", "endpoint": f"/rest/v1/{WANTED_TABLE}",
               "schema": "world_cup_arena"},
    description=f"Fetch {WANTED_TABLE} priors for both teams",
    input_payload={"country_id": f"in.({TEAM_A_ID},{TEAM_B_ID})", "select": "*"},
    output_payload=priors_rows,
    success=True,
)

# (9) Thinking — Supabase digest
rec_th_supabase = _new_record(
    "Thinking",
    upstream_record_id=[rec_sb_priors["record_id"]],
    model_invocation=_mi(llm_sb),
    prompt=_trunc(SUPABASE_DIGEST_SYS, limit=16000),
    inputs=[{
        "input_record_id": rec_sb_priors["record_id"],
        "input_payload":   _trunc({
            "fixture":      fixture["name"],
            "source_table": WANTED_TABLE,
            "home_code":    home["short_code"],
            "away_code":    away["short_code"],
            "rows":         priors_rows,
        }),
    }],
    output_payload=_trunc(supabase_digest),
)

# (10) Thinking — Predict (priors only, blind to market).
# The reasoning lives here; the structured prediction is committed via the
# Acting record below, which is the form the arena validates + scores.
rec_th_predict = _new_record(
    "Thinking",
    upstream_record_id=[rec_th_sportmonks["record_id"], rec_th_supabase["record_id"]],
    model_invocation=_mi(llm_predict),
    prompt=_trunc(PREDICT_SYS, limit=16000),
    inputs=[
        {"input_record_id": rec_th_sportmonks["record_id"],
         "input_payload":   _trunc(sportmonks_digest)},
        {"input_record_id": rec_th_supabase["record_id"],
         "input_payload":   _trunc(supabase_digest)},
    ],
    output_payload=_trunc(prediction),
)

# (11) Acting — Prediction (validated + scored by the arena).
# Per the new ledger contract, predictions are emitted as Acting records with
# action_type="prediction" and structured `parameters` the arena snapshots
# for scoring at settlement. probability is clamped to the schema range
# [0.001, 0.999].
_pred_prob = max(0.001, min(0.999, float(prediction["probability"])))
rec_act_predict = _new_record(
    "Acting",
    upstream_record_id=[rec_th_predict["record_id"]],
    action_type=     "prediction",
    target_system=   "arena",
    action_summary=  f"Predict {prediction['outcome']} @ p={_pred_prob:.2f} for fixture {SPORTMONKS_FIXTURE_ID}",
    parameters=      {
        "fixture_code": str(SPORTMONKS_FIXTURE_ID),
        "outcome":      prediction["outcome"],
        "probability":  _pred_prob,
    },
    dry_run=         False,
    execution_status="confirmed",
)

# (12) Thinking — Strategy (prediction + market → trade decision)
rec_th_strategy = _new_record(
    "Thinking",
    upstream_record_id=[rec_th_predict["record_id"], rec_th_polymarket["record_id"]],
    model_invocation=_mi(llm_strategy),
    prompt=_trunc(STRATEGY_SYS, limit=16000),
    inputs=[
        {"input_record_id": rec_th_predict["record_id"],
         "input_payload":   _trunc(prediction)},
        {"input_record_id": rec_th_polymarket["record_id"],
         "input_payload":   _trunc(polymarket_digest)},
    ],
    output_payload=_trunc(strategy),
)

records = [
    rec_trigger, rec_sm_schedule,
    rec_pm_slug, rec_pm_event, rec_pm_mids,
    rec_sm_fixture, rec_th_sportmonks,
    rec_th_polymarket,
    rec_sb_catalog, rec_sb_priors, rec_th_supabase,
    rec_th_predict, rec_act_predict, rec_th_strategy,
]

# (13) Acting — emit only when the agent actually submitted an order.
# This is the AGENT-side Acting (intent / submission). The arena will
# additionally write its own Acting record(s) server-side at fill / close
# time with target_system="public-chain" + execution_id=<tx_hash>. The two
# are independent evidence of the same logical action.
if strategy and strategy.get("should_trade"):
    # Did the order POST land cleanly? If yes, status=pending (waiting on fill);
    # if not, status=failed. order_response is None on 404 / exception.
    submitted_ok = isinstance(order_response, dict) and bool(order_response)
    rec_act = _new_record(
        "Acting",
        upstream_record_id=[rec_th_strategy["record_id"]],
        action_type=     "open_order",
        target_system=   "arena",     # we submit to arena; arena routes to polymarket-clob
        action_summary=  (f"Open {strategy['direction']} ${strategy['size_usdc']:.2f} on "
                          f"{strategy['outcome']} @ ≤{strategy['limit_price']}"),
        parameters=      order_payload,
        dry_run=         False,
        execution_status="pending" if submitted_ok else "failed",
        execution_id=    (order_response.get("order_id") if submitted_ok else None),
    )
    records.append(rec_act)

print()
print(f"=== STEP 8 · {len(records)} schema-compliant records ===")
for r in records:
    print(f"  {r['behavior']:12s}  {r['record_id'][:8]}…  "
          f"{r.get('description') or r.get('prompt','')[:40] or r.get('trigger_description','')[:40]}")

# Submit the trace as a single batch. Per the new ledger contract:
#   - No session-create endpoint; session_id is purely a client-side string.
#   - Bare record dicts (no {"body": {...}} envelope).
#   - agent_id is derived server-side from x-api-key.
#   - One round-trip per cycle via /records/batch (≤50 records). Response:
#       {"records": [<enriched echoes>], "errors": [{index, code, message}, ...]}
# Endpoint isn't live on staging yet — expect 404. Script reports rather than raises.
try:
    r = requests.post(
        f"{ARENA}/api/v1/arena/ledger/records/batch",
        headers=H_ARENA, timeout=15,
        json={"records": records},
    )
    print(f"\nledger batch POST  HTTP {r.status_code}  body: {r.text[:300]}")
    if r.ok:
        resp = r.json()
        print(f"  enriched: {len(resp.get('records', []))}  errors: {len(resp.get('errors', []))}")
        for e in resp.get("errors", []):
            print(f"    [#{e.get('index')}] {e.get('code')}: {e.get('message')}")
except Exception as e:
    print(f"ledger POST failed: {type(e).__name__}: {e}")
